# Sample Tracking

This notebook follows samples rather than measurements. Every other notebook in this repo reads `/entry` — the experimental records. This one reads `/sample`, the DMS's custody log: where each specimen has been, who handled it, and when.

It covers:

- **`GET /sample`** — the sample registry, and how little it carries on its own
- **`GET /sample/id`** — the per-sample event history, which is the only place the custody trail lives
- **Event types, handlers and locations** — what the free-text fields actually contain
- **The timeline** — one row per sample, coloured by who recorded the event

The timeline plot used to live in `imqcam.py` as `build_timeline()`. It is defined here instead: it is used by this notebook and nothing else, and inlining it separates the slow fetch from the cheap redraw.

## Setup

`get_client()` reads `GIRDER_API_KEY` from a `.env` at the repo root and pins the host to `https://data.imqcam.org/api/v1`.

The `REPO_ROOT` lookup walks up from the working directory until it finds `imqcam.py`, so the notebook runs the same from the repo root or from `tutorials/`.

In [1]:
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

# These notebooks are committed with outputs. The default renderer inlines the
# whole plotly.js bundle -- 4.5 MB into the .ipynb -- so load it from the CDN.
pio.renderers.default = "notebook_connected"

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client

# The shared palette, plus a fifth hue -- five people record events.
BLUE, ORANGE, GREEN, PURPLE, AMBER = "#4C8BE2", "#E26C4C", "#4CE2A7", "#9B7CE2", "#E2C44C"

client = get_client()

samples = client.get("sample", parameters={"limit": 1000})
print(f"{len(samples)} samples registered")
pd.DataFrame(samples).head()

377 samples registered


,_accessLevel,_id,_modelType,created,creator,description,eventTypes,name,updated
0,2,697cecda780c3cd3f8b2abef,sample,2026-01-30T17:39:38.926000+00:00,65b80ef4f4d3724f8bc3a004,None,[],TMABMX00001,2026-01-30T17:39:38.926000+00:00
1,2,6a592662e5ff4cfe370c8742,sample,2026-07-16T18:43:46.493000+00:00,65c5387402ad536bd833de56,None,[],JHABOX00001,2026-07-16T18:43:46.493000+00:00
2,2,6a0e19ec46a84d51432d067c,sample,2026-05-20T20:30:36.095000+00:00,67ad6270703d68f56a59a4e7,None,[],CMXMAL00026,2026-05-20T20:30:36.095000+00:00
3,2,6a0e179720913af2e8ee183c,sample,2026-05-20T20:20:39.528000+00:00,67ad6270703d68f56a59a4e7,None,[],CMXMAL00025-024,2026-05-20T20:20:39.528000+00:00
4,2,6a0e179720913af2e8ee183b,sample,2026-05-20T20:20:39.528000+00:00,67ad6270703d68f56a59a4e7,None,[],CMXMAL00025-023,2026-05-20T20:20:39.528000+00:00


## The event history is one request per sample

`/sample` returns only the envelope — name, creator, timestamps. The `events` list is on `/sample/id`, and there is no bulk endpoint for it, so collecting the custody trail is N+1 by necessity. At 377 samples this takes a couple of minutes and dominates the notebook's runtime.

It is also why the fetch is kept in its own cell: the plot below can be redrawn without paying for it again.

In [2]:
records = []
for sample in samples:
    detail = client.get("sample/id", parameters={"id": sample["_id"]})
    for event in detail.get("events", []):
        records.append(
            {
                "sample_id": sample["_id"],
                "sample_name": detail.get("name"),
                "event_type": event.get("eventType"),
                "creator": event.get("creatorName"),
                "comment": event.get("comment"),
                "timestamp": event.get("created"),
                "location": event.get("location"),
            }
        )

events = pd.DataFrame(records)
events["timestamp"] = pd.to_datetime(events["timestamp"], format="mixed", utc=False)

tracked = events["sample_name"].nunique()
print(f"{len(events)} events across {tracked} samples")
print(f"{len(samples) - tracked} of {len(samples)} samples have no events at all")

456 events across 159 samples
218 of 377 samples have no events at all


In [3]:
events = events[events["creator"] != "Kacper Kowalik"].copy()
events["creator"].value_counts().rename_axis("creator").to_frame("events")

,events
creator,
Sierra Green,234
Katie O'Donnell,122
Brett Ley,56
Kourtney Porsch,38
Brendan Croom,5


## What the events say

`eventType` and `location` are free text, and it shows. The same step is written several ways — `Returned from machining` against `Returned from Machining`, `testing in progress` against `testing completed` — and locations appear both spelled out and abbreviated (`Carnegie Mellon University` / `CMU`, `Case Western` / `CWRU`). Grouping on these fields without normalising them will split real categories.

In [4]:
summary = pd.concat(
    [
        events["event_type"].value_counts().rename_axis("value").to_frame("events").assign(field="event_type"),
        events["location"].value_counts().rename_axis("value").to_frame("events").assign(field="location"),
    ]
).reset_index()

summary[["field", "value", "events"]].head(20)

,field,value,events
0,event_type,Returned from machining,65
1,event_type,Shipped to CWRU from CMU,57
2,event_type,Sample Received,55
3,event_type,Stress Relief,50
4,event_type,Out for Machining 4/24/25,50
5,event_type,Returned from Machining,44
6,event_type,Heat Treatment,41
7,event_type,CWRU for Testing,39
8,event_type,testing in progress,28
9,event_type,EBSD,6


## The timeline

`px.timeline` needs a start and an end, but an event is an instant — the DMS records when a step happened, not how long it took. Each event is therefore drawn as a fixed one-day bar; the gaps between bars carry the meaning, not the bars themselves.

`filter_by_location` drops events with no usable site. Locations are free text, so blanks, `Unknown` and stray single letters all occur; anything shorter than two characters is not a site name.

In [5]:
def plot_timeline(events, filter_by_location=False):
    """Plot sample events over time, one row per sample.

    Args:
        events: DataFrame from the fetch above, with a datetime `timestamp`.
        filter_by_location: bool, drop events with no recorded location.
    Returns:
        The Plotly figure.
    """
    df = events.copy()
    if filter_by_location:
        location = df["location"].fillna("").astype(str).str.strip()
        df = df[(location.str.len() > 1) & (location.str.lower() != "unknown")]

    df = df.sort_values(["sample_name", "timestamp"])
    df["timestamp_end"] = df["timestamp"] + timedelta(days=1)

    applied = " (events with a recorded location)" if filter_by_location else ""
    fig = px.timeline(
        df,
        x_start="timestamp",
        x_end="timestamp_end",
        y="sample_name",
        color="creator",
        color_discrete_sequence=[BLUE, ORANGE, GREEN, PURPLE, AMBER],
        hover_data=["event_type", "comment", "location"],
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"Sample event timeline — n={len(df)} events, {df['sample_name'].nunique()} samples{applied}",
        xaxis_title="Date",
        yaxis_title="Sample",
        height=1200,
    )
    fig.show()
    return fig


_ = plot_timeline(events)

The bands are batches: samples move through machining, heat treatment and testing in groups, not individually. The horizontal runs are specimens that sat between steps, and each colour change on a row is a handoff between sites.

### Only events with a location

Restricting to events that name a site gives the shipping trail on its own — the same samples, minus the steps recorded without a destination.

In [6]:
_ = plot_timeline(events, filter_by_location=True)

## What this does not tell you

The custody log and the experimental records are separate systems. Samples here are named by IGSN, and `/entry` records carry IGSNs too (see `extract_igsn` in `imqcam.py`), so the two *can* be joined — but most registered samples have no events, and the event vocabulary is uncontrolled. Treat this as provenance context for a specimen you are already looking at, not as a dataset to aggregate over.

For the experimental records themselves, see `02_form_catalog.ipynb` and the notebooks in `analysis/`.